In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.simplefilter('ignore')

SEED = 30

In [2]:
df_test = pd.read_csv("/kaggle/input/playground-series-s5e6/test.csv")
df_train = pd.read_csv("/kaggle/input/playground-series-s5e6/train.csv")

In [3]:
def mapk(actual, predicted, k=3):
    total_score = 0.0
    actual = le.inverse_transform(actual)
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            index = p.index(a)
            total_score += 1.0 / (index + 1)
    return total_score / len(actual)

## Feature Engineering

In [4]:
df_train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [5]:
le = LabelEncoder()
le.fit(df_train['Fertilizer Name'])

AGGREGATES = ['mean']
FEATURES = ['Temparature', 'Humidity', 'Moisture', 'Soil Type',
            'Crop Type', 'Nitrogen', 'Potassium', 'Phosphorous']

def make_log(df, features=FEATURES):
    """
    Makes log transformations for all features in features
    """
    df_temp = df.copy()
    features = list(df_temp[features].select_dtypes(include=['int64', 'float64']).columns)

    for feature in features:
        df_temp[f'{feature}_log'] = np.log1p(df_temp[feature])
    return df_temp

def make_interactions(df, features=FEATURES):
    """
    Makes interactions between all features listed in features
    """
    df_temp = df.copy()
    cat_cols = list(df_temp[features].select_dtypes(include=['category']).columns)
    num_features = list(df_temp[features].select_dtypes(include=['int64', 'float64']).columns)

    df_temp2 = pd.get_dummies(df_temp, columns=cat_cols)
    cat_features = list(df_temp2.select_dtypes(include=['bool']).columns)
    features = num_features + cat_features

    for i in range(len(features) - 1):
        for j in range(i, len(features) - 1):
            df_temp[f'{features[i]}_{features[j]}'] = df_temp2[features[i]] * df_temp2[features[j]]

    return df_temp

def make_polynomials(df, features=FEATURES):
    """
    Make polynomial of features listed in features
    Only does 2nd and 3rd power for now
    """
    df_temp = df.copy()
    num_features = list(df_temp[features].select_dtypes(include=['int64', 'float64']).columns)

    for feature in num_features:
        df_temp[f'{feature}2'] = df_temp[feature] ** 2
        df_temp[f'{feature}3'] = df_temp[feature] ** 3

    return df_temp

def make_aggregates(df, features=FEATURES):
    """
    Makes aggregates of features listed in features using the formula
    new_feature = X - aggregate_type(df.groupby(Y)[X])
    - where X is a numerical feature and Y is a categorical feature
    - chosen aggregates are set in AGGREGATES
    """
    df_temp = df.copy()
    num_cols = list(df[features].select_dtypes(include=['int64', 'float64']).columns)
    cat_cols = list(df[features].select_dtypes(include=['category']).columns)

    for cat_col in cat_cols:
        for agg_type in AGGREGATES:
            aggs = df_temp[num_cols] - df_temp.groupby(cat_col)[num_cols].transform(agg_type)
            aggs.columns  = [f"{cat_col}_{num_col}_{agg_type}" for num_col in aggs.columns]
            df_temp = pd.concat([df_temp, aggs], axis=1)
    return df_temp

def make_features(df, test=False):
    df_temp = df.copy()
    df_temp.drop(columns=['id'], inplace=True)
    cat_cols = df_temp.select_dtypes(include=['object']).columns
    df_temp[cat_cols] = df_temp[cat_cols].astype('category')

    if not test:
        df_temp['Fertilizer Name'] = le.transform(df_temp['Fertilizer Name'])

    # df_temp = make_log(df_temp)
    # df_temp = make_interactions(df_temp)
    # df_temp = make_polynomials(df_temp)
    # df_temp = make_aggregates(df_temp)
    
    return df_temp

In [6]:
df_train1 = make_features(df_train)

In [7]:
initial_params = {
    "tree_method": "gpu_hist",
    "predictor": "gpu_predictor",
    'seed': SEED,
    'enable_categorical': True,
    'early_stopping_rounds': 100
}

In [8]:
X = df_train1.drop(columns=['Fertilizer Name'])
y = df_train1['Fertilizer Name']

In [9]:
def cross_val(X, y, params=initial_params, K=10, debug=True):
    kf = KFold(n_splits=K, shuffle=True, random_state=SEED)
    fold_scores = []
    fold = 0

    for train_idx, val_idx in kf.split(X):
        fold += 1
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBClassifier(
            **params
        )
        model.fit(X_train_fold, y_train_fold, eval_set=[(X_val_fold, y_val_fold)], verbose=False)

        val_pred = model.predict_proba(X_val_fold)
        val_pred = np.argsort(val_pred, axis=1)[:, -3:][:, ::-1]
        val_pred = [[le.classes_[j] for j in row] for row in val_pred]

        score = mapk(y_val_fold, val_pred)
        fold_scores.append(score)
        if debug == True:
            print(f'Fold {fold} Mean Average Precision Score: {score}')

    avg_score = np.mean(fold_scores)
    if debug == True:
        print(f'Average Validation Fold Score: {avg_score}')
    return avg_score

In [10]:
xgb_params1 = {
    'learning_rate': 0.03,
     'max_depth': 12,
     'subsample': 0.30000000000000004,
     'colsample_bytree': 0.30000000000000004,
     'max_bin': 926, 
     'min_child_weight': 6,
     'gamma': 0.1,
     'lambda': 1.8875729586599337,
     'alpha': 0.004494936589076713,
     'max_delta_step': 1,
    'n_estimators': 3000,
    'enable_categorical': True,
    'early_stopping_rounds': 100,
    'random_state': SEED,
    'tree_method': 'gpu_hist',
    'preditor': 'gpu_predictor'
}

xgb_params2 = {'learning_rate': 0.03,
               'max_depth': 11,
               'subsample': 0.8,
               'colsample_bytree': 0.30000000000000004,
               'max_bin': 1551,
               'min_child_weight': 3,
               'gamma': 0.0,
               'lambda': 0.0013312723042592412,
               'alpha': 0.6136573473631746,
               'max_delta_step': 8,
               'n_estimators': 3000,
               'enable_categorical': True,
               'early_stopping_rounds': 100,
               'random_state': SEED,
               'tree_method': 'gpu_hist',
               'preditor': 'gpu_predictor'
              }

xgb_params3 = {
                'objective': 'multi:softprob',
                'eval_metric': 'mlogloss',
                'eta': 0.01,
                'max_depth': 10,
                'subsample': 0.7,
                'colsample_bytree': 0.5,
                'n_estimators': 10000,
                'random_state': 30,
                'tree_method': 'gpu_hist',
                'predictor': 'gpu_predictor',
                'n_jobs': -1,
                'early_stopping_rounds': 50,
                'verbose': 0,
                'enable_categorical': True
            }

# this stuff from https://www.kaggle.com/code/pirhosseinlou/xgboost-single-model-v2
xgb_params4 = {
        'objective': 'multi:softprob',
        'num_class': 7,
        'max_depth': 12,
        'learning_rate': 0.02,
        'n_estimators': 100_000,
        'reg_alpha': 3,
        'reg_lambda': 1.4,
        'gamma': 0.26,
        'max_delta_step': 5,
        'subsample': 0.86,
        'colsample_bytree': 0.4,
        'min_child_weight': 5,
        'random_state': 42,
        'n_jobs': -1,
        'early_stopping_rounds': 30,
        'eval_metric': 'mlogloss',
        'enable_categorical': True,
        'device': "cuda"
    }

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [12]:
K = 10
kf = KFold(n_splits=K, random_state=SEED, shuffle=True)

base_models = [
    XGBClassifier(**xgb_params1),
    XGBClassifier(**xgb_params2),
    XGBClassifier(**xgb_params3)
    # XGBClassifier(**xgb_params4)
             ]
N_MODELS = len(base_models)
NUM_CLASSES = 7

oof_train = np.zeros((len(X_train), N_MODELS * NUM_CLASSES))
val_preds = np.zeros((len(X_val), N_MODELS * NUM_CLASSES))

for m_idx, model in enumerate(base_models):
    val_fold_preds = []
    scores = []
    fold = 0
    model_name = None
    for train_idx, val_idx in kf.split(X_train, y_train):
        X_tr, X_v = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_v = y_train.iloc[train_idx], y_train.iloc[val_idx]
        fold += 1

        model_name = 'XGB'
        model.fit(X_tr, y_tr, eval_set=[(X_v, y_v)], verbose=False)

        # Predict on validation fold and store in OOF matrix
        probas = model.predict_proba(X_v)
        oof_train[val_idx, m_idx*NUM_CLASSES:(m_idx+1)*NUM_CLASSES] = probas

        # Predict on test data and save for averaging
        val_pred = model.predict_proba(X_val)
        val_fold_preds.append(val_pred)
        val_pred = np.argsort(val_pred, axis=1)[:, -3:][:, ::-1]
        val_pred = [[le.classes_[j] for j in row] for row in val_pred]
        score = mapk(y_val, val_pred)
        scores.append(score)
        print(f"{model_name} fold {fold} val score: {score}")

    avg_score = np.mean(scores)
    print(f"XGB {m_idx} Average Val Score: {avg_score}")
    val_preds[:, m_idx*NUM_CLASSES:(m_idx+1)*NUM_CLASSES] = np.mean(val_fold_preds, axis=0)

XGB fold 1 val score: 0.35384666666676295
XGB fold 2 val score: 0.3544233333334266
XGB fold 3 val score: 0.3555633333334296
XGB fold 4 val score: 0.355293333333429
XGB fold 5 val score: 0.35567111111120603
XGB fold 6 val score: 0.35406222222231637
XGB fold 7 val score: 0.35458555555565296
XGB fold 8 val score: 0.3552122222223195
XGB fold 9 val score: 0.3548211111112059
XGB fold 10 val score: 0.35505666666676267
XGB 0 Average Val Score: 0.3548535555556511
XGB fold 1 val score: 0.3548377777778756
XGB fold 2 val score: 0.35585444444454
XGB fold 3 val score: 0.35588444444454226
XGB fold 4 val score: 0.35550666666676434
XGB fold 5 val score: 0.3568044444445449
XGB fold 6 val score: 0.3557300000000948
XGB fold 7 val score: 0.3560777777778756
XGB fold 8 val score: 0.35590888888898475
XGB fold 9 val score: 0.355041111111206
XGB fold 10 val score: 0.35642666666676615
XGB 1 Average Val Score: 0.3558072222223194
XGB fold 1 val score: 0.34611888888898085
XGB fold 2 val score: 0.3463577777778666
XG

## Shoving Stuff into Logistic Regression Model

In [13]:
from sklearn.linear_model import LogisticRegression


In [14]:
meta_params = {
    'C': 0.01,
    'max_iter': 100,
    'penalty': 'l2',
    'solver': 'saga'
}

In [15]:
# training meta model
meta_model = LogisticRegression(
    **meta_params
    )

# Fit on out-of-fold predictions
meta_model.fit(oof_train, y_train)

# Predict on held-out validation set
meta_pred = meta_model.predict_proba(val_preds)
meta_pred = np.argsort(meta_pred, axis=1)[:, -3:][:, ::-1]
meta_pred = [[le.classes_[j] for j in row] for row in meta_pred]

score = mapk(y_val, meta_pred)
print("Meta-model validation MAP@3 Score:", score)

Meta-model validation MAP@3 Score: 0.3579977777778749


## Submission

In [16]:
df_test1 = make_features(df_test, test=True)

In [17]:
model1 = XGBClassifier(**xgb_params1)
model2 = XGBClassifier(**xgb_params2)
model3 = XGBClassifier(**xgb_params3)

model1.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
model2.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
model3.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

test_preds = np.zeros((len(df_test1), N_MODELS * NUM_CLASSES))
test_preds[:, 0 * NUM_CLASSES:1*NUM_CLASSES] = model1.predict_proba(df_test1)
test_preds[:, 1 * NUM_CLASSES:2*NUM_CLASSES] = model2.predict_proba(df_test1)
test_preds[:, 2 * NUM_CLASSES:3*NUM_CLASSES] = model3.predict_proba(df_test1)

In [18]:
y_test_pred = meta_model.predict_proba(test_preds)
y_test_pred = np.argsort(y_test_pred, axis=1)[:, -3:][:, ::-1]
y_test_pred = [[le.classes_[j] for j in row] for row in y_test_pred]
y_test_pred = [' '.join(row) for row in y_test_pred]

submission = pd.read_csv("/kaggle/input/playground-series-s5e6/sample_submission.csv")
submission['Fertilizer Name'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Fertilizer Name
0,750000,10-26-26 20-20 28-28
1,750001,17-17-17 20-20 28-28
2,750002,20-20 28-28 Urea
3,750003,14-35-14 10-26-26 17-17-17
4,750004,20-20 Urea 28-28
